# OPE bootstrap and Jacobi identities

A universal vertex algebra is given by strong generators and their OPEs, with `UniversalVertexAlgebra(R, generators, opes, ...)` (`01_getting_started.ipynb`, section 2). The OPEs define a vertex algebra, with the ordered normally ordered monomials in the generators and their derivatives as a basis, exactly when they satisfy the Jacobi identities [DSK2005]. This notebook shows two tools built on this criterion:

* **Checking declared OPEs** (section 1). `V.jacobi_identity_failures()` lists the Jacobi identities that fail, `V.satisfies_jacobi_identity()` tests that none fails, and `UniversalVertexAlgebra(..., verify_jacobi=True)` refuses inconsistent OPEs.
* **Solving for unknown OPEs** (sections 2–7). For `OPEBootstrap(V)`, the generators of `V` are given conformal weights (`degrees=`), and only the OPEs that are known are declared, for example the Virasoro OPE and the OPEs of primary fields with the Virasoro field. Every undeclared OPE gets the most general ansatz allowed by the conformal weights, the parities and, if one is set, a grading, with unknown coefficients `x0, x1, ...`. The Jacobi identities become polynomial equations in the unknowns. `impose_jacobi()` imposes them one triple of generators at a time, eliminating unknowns as it goes; `solve()` returns the irreducible components of the set of solutions, and `S.algebra()` is the vertex algebra of a component `S`.

This is the method by which W-algebras with few generators were classified [KW1991, BFKNRV1991]; Thielemans' Mathematica package [Thi1991] is a standard tool for the OPE computations involved.

**Coefficients.** Over a field, such as `QQ` or `Frac(QQ['c'])`, the central charge $c$ is generic. Over a polynomial ring, such as `QQ['c']`, $c$ is a parameter that is solved for like the unknowns, so special values of $c$ show up as separate components (section 3).

**Prerequisites:** `01_getting_started.ipynb`.

**Elsewhere:** W-algebras by quantum Hamiltonian reduction, with OPEs computed rather than bootstrapped, are in `08_w_algebras.ipynb`; modules over the $W_3$ algebra of section 1 are in `10_modules.ipynb`, section 4; the presentation of a subalgebra whose OPEs satisfy the Jacobi identities only modulo null fields is in `15_triplet_modules.ipynb`, section 2.

**Contents**

1. Checking declared OPEs: the $W_3$ algebra
2. Bootstrapping $W_3$
3. The central charge as a parameter
4. Square roots: $W(2,4)$
5. The $N=2$ superconformal algebra: odd generators and a grading
6. $W(2,3,4)$
7. Saving and resuming a bootstrap
8. Limitations

In [1]:
import vertex_algebras as va
from vertex_algebras import *
import os, sys, time
from vertex_algebras.universal_vertex_algebra import UniversalVertexAlgebra
from vertex_algebras.normally_ordered_expressions import NormallyOrderedExpressions
from vertex_algebras.bootstrap import OPEBootstrap

## 1. Checking declared OPEs: the $W_3$ algebra

Zamolodchikov's $W_3$ algebra [Zam1985] is generated by the Virasoro field $L$ and a primary field $W$ of weight 3, with the OPE

$$W(z)W(w) \sim \frac{c/3}{(z-w)^6} + \frac{2L(w)}{(z-w)^4} + \frac{\partial L(w)}{(z-w)^3} + \frac{2\beta\Lambda(w) + \frac{3}{10}\partial^2 L(w)}{(z-w)^2} + \frac{\beta\partial\Lambda(w) + \frac{1}{15}\partial^3 L(w)}{z-w},$$

where $\Lambda = {:}LL{:} - \frac{3}{10}\partial^2 L$. These OPEs are consistent exactly for $\beta = 16/(22+5c)$. The coefficients are written as normally ordered expressions in $L$ and $W$ (words of fields, nested to the right, see `01_getting_started.ipynb`, section 2) before the algebra exists. The constructor then normally orders them with the OPEs themselves: for example, $\partial\Lambda$ contains ${:}L\,\partial L{:}$, whose expression in the basis needs the OPE of $L$ with $L$.

The cell below defines the OPEs as a function `w3(beta)`, checks the Jacobi identities for $\beta = 16/(22+5c)$, and prints the number of failures and the first three failures for $\beta = 1$.

In [2]:
K.<c> = Frac(QQ['c'])
E.<L, W> = NormallyOrderedExpressions(K)
VIR = {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'W'): [W.derivative(), 3*W]}
Lam = L*L - 3/10*L.derivative(2)

def w3(beta):
    WW = [L.derivative(3)/15 + beta*Lam.derivative(), 3/10*L.derivative(2) + 2*beta*Lam,
          L.derivative(), 2*L, 0, c/3]
    return dict(VIR) | {('W', 'W'): WW}

V = UniversalVertexAlgebra(K, E, w3(16/(22 + 5*c)))
print('beta = 16/(22+5c):', V.satisfies_jacobi_identity())
V = UniversalVertexAlgebra(K, E, w3(1))
bad = V.jacobi_identity_failures()
print('beta = 1:', len(bad), 'failing identities, e.g.')
for label in list(bad)[:3]:
    print('   ', label, ':', bad[label])

beta = 16/(22+5c): True
beta = 1: 18 failing identities, e.g.
    ('jacobi', 'L', 'W', 'W', 3, 0) : (c + 6/5)*dL
    ('jacobi', 'L', 'W', 'W', 3, 1) : (2*c + 12/5)*L
    ('jacobi', 'L', 'W', 'W', 4, 0) : (4*c + 24/5)*L


The failures are labelled in two ways. The label `('jacobi', a, b, d, m, n)` stands for the commutator formula

$$a_{(m)}(b_{(n)}d) - (-1)^{|a||b|}\, b_{(n)}(a_{(m)}d) - \sum_{j \ge 0} \binom{m}{j}(a_{(j)}b)_{(m+n-j)}d = 0$$

for generators $a, b, d$, and the value printed is the left-hand side. The label `('skew', a, b, n)` stands for the skew-symmetry of $a_{(n)}b$, which is checked for the OPEs declared directly (a generator with itself, or a pair declared in both orders). With `verify_jacobi=True`, the constructor refuses inconsistent OPEs, as the cell below shows.

In [3]:
try:
    UniversalVertexAlgebra(K, E, w3(1), verify_jacobi=True)
except ValueError as e:
    print(str(e)[:200])
print(UniversalVertexAlgebra(K, E, w3(16/(22 + 5*c)), verify_jacobi=True).satisfies_jacobi_identity())

the OPEs violate the Jacobi identity [L_(3), W_(0)] W: (c + 6/5)*dL (and at least 2 more)
True


## 2. Bootstrapping $W_3$

Here only the OPEs of $L$ with $L$ and of $L$ with $W$ are declared, and the generators are given the weights 2 and 3. `OPEBootstrap(V)` writes the most general ansatz for the OPE of $W$ with $W$. The first cell below prints the bootstrap, the ansatz, and the unknowns that remain, with the meaning of each given by `describe(x)`. The ansatz has 11 unknowns, and skew-symmetry removes 6 of them at once, which leaves `x0, x1, x4, x5, x6`.

The second cell fixes the normalisation `x0`, the coefficient of the vacuum in $W_{(5)}W$, to $c/3$, and imposes the Jacobi identities. The triples containing $L$ express the Virasoro covariance of the ansatz. Here the triple $(L, W, W)$ determines all the remaining unknowns, and the triple $(W, W, W)$ then holds without further conditions. The solution is Zamolodchikov's $W_3$ algebra: the coefficient of ${:}LL{:}$ in $W_{(1)}W$ is $\frac{32/5}{c + 22/5} = 2\beta$.

In [4]:
V = UniversalVertexAlgebra(K, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'W'): [W.derivative(), 3*W]},
                           degrees={'L': 2, 'W': 3})
B = OPEBootstrap(V)
print(B)
print(B.ansatz())
print([(x, B.describe(x)) for x in B.unknowns()])

OPE bootstrap of Universal vertex algebra on generators ['L', 'W'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field: 11 unknowns in the OPEs of [('W', 'W')] (5 left), 0 equations, Jacobi identities imposed for 0 of 8 triples
{('W', 'W'): [x4*:dL L: + (-1/24*x1 + 1/12*x4 + 1/2*x5)*d^3L + 1/2*x6*d^2W, x4*:L L: + x5*d^2L + x6*dW, 1/2*x1*dL, x1*L, 0, x0]}
[(x0, 'coefficient of 1 in W_(5) W'), (x1, 'coefficient of L in W_(3) W'), (x4, 'coefficient of :L L: in W_(1) W'), (x5, 'coefficient of d^2L in W_(1) W'), (x6, 'coefficient of dW in W_(1) W')]


In [5]:
B.fix({'x0': c/3})
B.impose_jacobi(verbose=True)
[S] = B.solve()
A = S.algebra()
for k, p in enumerate(A.opes()[('W', 'W')]):
    print('W_(%d) W =' % k, p)
print(A.satisfies_jacobi_identity())

Jacobi (L, L, L): 0 failing identities, 4 unknowns left, 0 equations, 0.0 s


Jacobi (L, L, W): 0 failing identities, 4 unknowns left, 0 equations, 0.0 s


Jacobi (L, W, L): 0 failing identities, 4 unknowns left, 0 equations, 0.0 s


Jacobi (W, L, L): 0 failing identities, 4 unknowns left, 0 equations, 0.0 s


Jacobi (L, W, W): 9 failing identities, 0 unknowns left, 0 equations, 0.0 s


Jacobi (W, L, W): 0 failing identities, 0 unknowns left, 0 equations, 0.0 s


Jacobi (W, W, L): 0 failing identities, 0 unknowns left, 0 equations, 0.0 s


Jacobi (W, W, W): 0 failing identities, 0 unknowns left, 0 equations, 0.0 s


W_(0) W = (32/5/(c + 22/5))*:dL L: + ((1/15*c - 2/15)/(c + 22/5))*d^3L
W_(1) W = (32/5/(c + 22/5))*:L L: + ((3/10*c - 3/5)/(c + 22/5))*d^2L
W_(2) W = dL
W_(3) W = 2*L
W_(4) W = 0
W_(5) W = 1/3*c
True


## 3. The central charge as a parameter

Over `QQ['c']`, the central charge $c$ is solved for like the unknowns. The cell below repeats the bootstrap of section 2 over `QQ['c']`, without fixing a normalisation. There is a single component, in which $c$ and the coefficient `x4` of ${:}LL{:}$ in $W_{(1)}W$ are free, and the normalisation `x0` is determined by them:

$$x_0 = \tfrac{1}{96}\, x_4\, c\, (5c + 22).$$

So for $x_4 \neq 0$ the two-point function `x0` vanishes at $c = 0$ and at $c = -\frac{22}{5}$, the central charge of the Lee–Yang model, at which $\Lambda$ is a null field. Correspondingly, in the normalisation $x_0 = c/3$ of section 2, the coefficient $x_4 = 2\beta = 32/(5c + 22)$ has a pole at $c = -\frac{22}{5}$.

In [6]:
P.<c> = QQ['c']
E.<L, W> = NormallyOrderedExpressions(P)
V = UniversalVertexAlgebra(P, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'W'): [W.derivative(), 3*W]},
                           degrees={'L': 2, 'W': 3})
B = OPEBootstrap(V)
B.impose_jacobi()
print(B.solve())
print('x0 =', B.eliminated()[B.ring()('x0')].factor())

[Bootstrap solution 1 of 1: free [c, x4]]
x0 = (1/96) * x4 * c * (5*c + 22)


**A primary field of weight 5.** The algebra $W(2,5)$ is generated by $L$ and a primary field $X$ of weight 5, here normalised by $X_{(9)}X = c/5$. The cell below imposes the Jacobi identities over `QQ['c']`. The remaining equation is $c^3 - 268c^2 - 44c = 0$, with two components: $c = 0$, where the normalisation $c/5$ vanishes, and $c^2 - 268c - 44 = 0$, that is, $c = 134 \pm 60\sqrt5$. So $W(2,5)$ exists as a universal vertex algebra, without null fields, only at these two values of $c$ [KW1991, BFKNRV1991]. The algebra of the second component is defined over the number field $\mathbb Q[c]/(c^2 - 268c - 44)$, and `algebra()` constructs it there.

In [7]:
E.<L, X> = NormallyOrderedExpressions(P)
V = UniversalVertexAlgebra(P, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'X'): [X.derivative(), 5*X]},
                           degrees={'L': 2, 'X': 5})
B = OPEBootstrap(V)
B.fix({'x0': c/5})
t = time.time(); B.impose_jacobi(); print('%.1f s' % (time.time() - t))
print(B.equations())
for S in B.solve():
    print(S)
S = B.solve()[0]
A = S.algebra()
print(A.base_ring(), A.satisfies_jacobi_identity())

0.2 s
[c^3 - 268*c^2 - 44*c]
Bootstrap solution 1 of 2: no free unknowns, c^2 - 268*c - 44 = 0
Bootstrap solution 2 of 2: no free unknowns, c = 0


Number Field in c with defining polynomial c^2 - 268*c - 44 True


## 4. Square roots: $W(2,4)$

For $W(2,4)$, with a primary field $W$ of weight 4 normalised by $W_{(7)}W = c/4$, the self-coupling of $W$, the coefficient `x5` of $W$ in $W_{(3)}W$, is

$$x_5 = \pm\sqrt{\frac{54(c+24)(c^2-172c+196)}{(5c+22)(7c+68)(2c-1)}}$$

[KW1991, BFKNRV1991]. The component of the solutions is therefore rational only over a quadratic extension of $\mathbb Q(c)$, and `algebra()` works over this extension. The cell below prints the unknown that is algebraic over $\mathbb Q(c)$, the factorisation of its square (the value $-p(0)$, for $p$ the minimal polynomial of `x5`), and checks the Jacobi identities of the algebra.

In [8]:
K.<c> = Frac(QQ['c'])
E.<L, W> = NormallyOrderedExpressions(K)
V = UniversalVertexAlgebra(K, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'W'): [W.derivative(), 4*W]},
                           degrees={'L': 2, 'W': 4})
B = OPEBootstrap(V)
B.fix({'x0': c/4})
B.impose_jacobi()
[S] = B.solve()
print(S.algebraic_variable(), '=', B.describe(S.algebraic_variable()))
p = S.extension().modulus()
print((-p(0)).factor())
print(S.algebra().satisfies_jacobi_identity())

x5 = coefficient of W in W_(3) W
(27/35) * (c - 1/2)^-1 * (c + 22/5)^-1 * (c + 68/7)^-1 * (c + 24) * (c^2 - 172*c + 196)


True


## 5. The $N=2$ superconformal algebra: odd generators and a grading

The generators are the Virasoro field $L$, a current $J$ of weight 1 and two odd primary fields $G^\pm$ of weight $\frac32$; odd generators are declared with `parities=` (`01_getting_started.ipynb`, section 3). The grading by the $U(1)$ charge, with $G^\pm$ of charge $\pm 1$ (`horizontal_grading=`), keeps the ansatz small, since every term of an OPE must have the total charge of the two fields. The cell below prints the ansatz and the components of the solutions. There are three:

1. `x5 = x7 = 0`: the OPE of $G^+$ with $G^-$ vanishes;
2. `x0 = c/3*x2^2, x3 = -x2, x7 = 0`, with `x2` and `x5` free: the $N = 2$ superconformal algebra, in which `x2` and `x5` are the normalisations of $J$ and of the OPE of $G^+$ with $G^-$. For `x2 = 1`, $J(z)J(w) \sim \frac{c/3}{(z-w)^2}$ and $J(z)G^\pm(w) \sim \pm \frac{G^\pm(w)}{z-w}$;
3. `x0 = x2 = x3 = 0`: the current $J$ has a vanishing two-point function and commutes with $G^\pm$.

In [9]:
E.<L, J, Gp, Gm> = NormallyOrderedExpressions(K, parities={'Gp': 1, 'Gm': 1})
V = UniversalVertexAlgebra(K, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'J'): [J.derivative(), J],
                                  ('L', 'Gp'): [Gp.derivative(), 3/2*Gp], ('L', 'Gm'): [Gm.derivative(), 3/2*Gm]},
                           degrees={'L': 2, 'J': 1, 'Gp': 3/2, 'Gm': 3/2},
                           horizontal_grading={'Gp': 1, 'Gm': -1})
B = OPEBootstrap(V)
B.impose_jacobi()
print(B.ansatz())
for S in B.solve():
    print(S)

{('J', 'J'): [0, x0], ('J', 'Gp'): [x2*Gp], ('J', 'Gm'): [x3*Gm], ('Gp', 'Gp'): [], ('Gp', 'Gm'): [(x2*x5 - 2*x0*x7)*L + x7*:J J: + 1/2*x5*dJ, x5*J, c/3*x2*x5 + (-2*c + 2)/3*x0*x7], ('Gm', 'Gm'): []}
Bootstrap solution 1 of 3: free [x0, x2, x3], x5 = 0, x7 = 0
Bootstrap solution 2 of 3: free [x2, x5], x0 = c/3*x2^2, x3 = -x2, x7 = 0
Bootstrap solution 3 of 3: free [x5, x7], x0 = 0, x2 = 0, x3 = 0


## 6. $W(2,3,4)$

With generators of weights 2, 3 and 4, there are three ansatz OPEs ($W$ with $W$, $W$ with $U$, and $U$ with $U$), with 73 unknowns in total, and 27 triples of generators. The cell below fixes the two-point functions $W_{(5)}W = c/3$ and $U_{(7)}U = c/4$ (the unknown for the second is found with `describe`) and imposes the Jacobi identities, which takes under a second here. The coupling `x7`, the coefficient of $U$ in $W_{(1)}W$, is then determined up to sign: the equation printed is

$$x_7^2 = \frac{16\,(c+2)(7c+114)}{3\,(c+7)(5c+22)}.$$

This is the W-algebra $W_4$ of $\mathfrak{sl}_4$ [KW1991, BFKNRV1991].

In [10]:
E.<L, W, U> = NormallyOrderedExpressions(K)
V = UniversalVertexAlgebra(K, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'W'): [W.derivative(), 3*W],
                                  ('L', 'U'): [U.derivative(), 4*U]},
                           degrees={'L': 2, 'W': 3, 'U': 4})
B = OPEBootstrap(V)
NU = [x for x in B.unknowns() if B.describe(x) == 'coefficient of 1 in U_(7) U'][0]
B.fix({'x0': c/3, NU: c/4})
t = time.time(); B.impose_jacobi(); print('%.1f s' % (time.time() - t))
for S in B.solve():
    print(S)

0.7 s
Bootstrap solution 1 of 1: no free unknowns, x7^2 + (-112/15*c^2 - 2048/15*c - 1216/5)/(c^2 + 57/5*c + 154/5) = 0


## 7. Saving and resuming a bootstrap

Large bootstraps can take a long time, so their state can be saved. `save(B, fn)` writes the algebra, the ansatz, the eliminated unknowns with their values, the remaining equations and the imposed triples to a file, and `impose_jacobi(..., save=fn)` does so after every triple. `load(fn)` returns a bootstrap that continues from the saved state, also in another session or on another machine: the file does not depend on the number of threads or processes. Loading checks that the saved state belongs to the same algebra and ansatz.

The cell below is a first session: it bootstraps $W(2,3,4)$ without normalisations, imposes only the triples of total weight at most 10, and saves the state after each one.

In [11]:
import tempfile
fn = os.path.join(tempfile.mkdtemp(), 'w234_bootstrap.sobj')   # any path

WTS = {'L': 2, 'W': 3, 'U': 4}

def w234():
    E.<L, W, U> = NormallyOrderedExpressions(K)
    return UniversalVertexAlgebra(K, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'W'): [W.derivative(), 3*W],
                                         ('L', 'U'): [U.derivative(), 4*U]},
                                  degrees=WTS)

V = w234()
B = OPEBootstrap(V)
cheap = [(a, b, d) for a in WTS for b in WTS for d in WTS if WTS[a] + WTS[b] + WTS[d] <= 10]
B.impose_jacobi(cheap, save=fn)
print(B)

OPE bootstrap of Universal vertex algebra on generators ['L', 'W', 'U'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field: 73 unknowns in the OPEs of [('W', 'W'), ('W', 'U'), ('U', 'U')] (5 left), 4 equations, Jacobi identities imposed for 23 of 27 triples


`load(fn)` returns a new bootstrap; in a new session, it also rebuilds `V` from the saved data. The loaded bootstrap knows which triples were imposed, and `impose_jacobi()` continues with the others. Without normalisations there are two components: the $W_4$ family of section 6, with `x7` and `x55` free, and a degenerate component with `x7 = 0`.

In [12]:
B2 = load(fn)
print(B2)
print(len(B2.imposed_triples()), 'triples imposed,', len(B2.unknowns()), 'unknowns left')
t = time.time(); B2.impose_jacobi(verbose=True); print('%.1f s' % (time.time() - t))
for S in B2.solve():
    print(S)

OPE bootstrap of Universal vertex algebra on generators ['L', 'W', 'U'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field: 73 unknowns in the OPEs of [('W', 'W'), ('W', 'U'), ('U', 'U')] (5 left), 4 equations, Jacobi identities imposed for 23 of 27 triples
23 triples imposed, 5 unknowns left


Jacobi (W, U, U): 46 failing identities, 3 unknowns left, 1 equations, 1.1 s


Jacobi (U, W, U): 46 failing identities, 3 unknowns left, 1 equations, 0.6 s


Jacobi (U, U, W): 37 failing identities, 3 unknowns left, 1 equations, 0.4 s


Jacobi (U, U, U): 50 failing identities, 3 unknowns left, 1 equations, 0.6 s


2.7 s
Bootstrap solution 1 of 2: free [x7, x55], x32 = 0
Bootstrap solution 2 of 2: free [x32, x55], x7 = 0


An uninterrupted run gives the same components:

In [13]:
B3 = OPEBootstrap(w234())
B3.impose_jacobi()
print([str(S) for S in B3.solve()] == [str(S) for S in B2.solve()])

True


## 8. Limitations

* **Universal vertex algebras only.** The ansatz is written in the normally ordered monomials of the generators, which are taken to be linearly independent, and `algebra()` is a universal vertex algebra with these monomials as a basis. Algebras whose OPEs satisfy the Jacobi identities only modulo null fields are not found, such as $W(2,5)$ at $c = -7$ [BFKNRV1991]. Such algebras can be studied as presentations of subalgebras (`06_subalgebras_ideals_quotients.ipynb`, section 6, and `15_triplet_modules.ipynb`, section 2).
* **Polynomial systems.** The equations that remain after elimination are decomposed into irreducible components (minimal associated primes, computed by Singular). With many unknowns left, this can be slow; fixing normalisations and setting a grading reduce the work.
* **Generic parameters.** Over `Frac(QQ['c'])` a solution is valid for generic $c$. Special values of $c$, where the solutions can be different, are found only over `QQ['c']` (section 3).
* **Normalisations.** The bootstrap does not fix normalisations: free unknowns in a component can be rescalings of the generators (as `x2` and `x5` in section 5) rather than genuine parameters, and the bootstrap does not distinguish the two. `B.fix` sets unknowns to chosen values.
* **Cost.** Each triple needs the OPEs up to the total weight of the triple, so the cost grows quickly with the weights of the generators; section 7 shows how to save and resume long bootstraps.

## References

- [DSK2005] A. De Sole, V. G. Kac, *Freely generated vertex algebras and non-linear Lie conformal algebras*, Comm. Math. Phys. 254 (2005), 659–694, [arXiv:math-ph/0312042](https://arxiv.org/abs/math-ph/0312042). — the Jacobi identities as the condition for a universal vertex algebra with a PBW basis (introduction).
- [KW1991] H. G. Kausch, G. M. T. Watts, *A study of W-algebras using Jacobi identities*, Nuclear Phys. B 354 (1991), 740–768, [doi:10.1016/0550-3213(91)90375-8](https://doi.org/10.1016/0550-3213(91)90375-8). — bootstrapping W-algebras with Jacobi identities (sections 3, 4, 6).
- [BFKNRV1991] R. Blumenhagen, M. Flohr, A. Kliem, W. Nahm, A. Recknagel, R. Varnhagen, *W-algebras with two and three generators*, Nuclear Phys. B 361 (1991), 255–289, [doi:10.1016/0550-3213(91)90624-7](https://doi.org/10.1016/0550-3213(91)90624-7). — $W(2,4)$, $W(2,5)$, $W(2,3,4)$ (sections 3, 4, 6, 8).
- [Thi1991] K. Thielemans, *A Mathematica package for computing operator product expansions*, Internat. J. Modern Phys. C 2 (1991), 787–798, [doi:10.1142/S0129183191001001](https://doi.org/10.1142/S0129183191001001). — computer algebra for OPEs (introduction).
- [Zam1985] A. B. Zamolodchikov, *Infinite additional symmetries in two-dimensional conformal quantum field theory*, Theoret. and Math. Phys. 65 (1985), 1205–1213, [doi:10.1007/BF01036128](https://doi.org/10.1007/BF01036128). — the $W_3$ algebra (sections 1–3).